In [1]:
import numpy as np
import pandas as pd

In [ ]:
import json


def get_title(anime):
    return anime.get("title_english") or anime.get("title")

def get_year(anime):
    return (
        anime.get("year")
        or (anime.get("aired", {}).get("from") or "")[:4] or None
    )

def get_rank():
    return


def get_json_anime_data(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        anime_entries = json.load(f)

    results = []
    # print(anime_entries)

    for anime in anime_entries:
        # print(type(anime))
        results.append({
            "mal_id": anime.get("mal_id"),
            "url": anime.get("url"),
            # "title_english": anime.get("title_english"),
            "title_english": get_title(anime),
            "type": anime.get("type"),
            "source": anime.get("source"),
            "episodes": anime.get("episodes"),
            "aired": anime.get("aired", {}).get("string"),
            "duration": anime.get("duration"),
            "rating": anime.get("rating"),
            "score": anime.get("score"),
            "scored_by": anime.get("scored_by"),
            "rank": anime.get("rank"),
            "popularity": anime.get("popularity"),
            "members": anime.get("members"),
            "favorites": anime.get("favorites"),
            "year": get_year(anime),
            "demographics": [
                d.get("name")
                for d in anime.get("demographics", [])
            ],

            "studios": [
                studio.get("name")
                for studio in anime.get("studios", [])
            ],

            "genres": [
                genre.get("name")
                for genre in anime.get("genres", [])
            ],

            "explicit_genres": [
                genre.get("name")
                for genre in anime.get("explicit_genres", [])
            ],

            "themes": [
                theme.get("name")
                for theme in anime.get("themes", [])
            ]
        })

    return results


def get_json_studios_data(filepath):
    # from test_anime.json
    # 'studios' -> ...

    with open(filepath, "r", encoding="utf-8") as f:
        anime_entries = json.load(f)

    studios = []  # 1 anime can have more than 1 studio, 1->N relation?

    for anime in anime_entries:
        for studio in anime.get("studios", []):
            studios.append({
                "studio_id": studio.get("mal_id"),  # .../producers/{mal_id}
                "studio_name": studio.get("name"),
                "studio_url": studio.get("url"),
                # "anime_id": anime.get('mal_id')  # think about it later
            })

    return studios


def json_unify(data, key):
    return list({
        item.get(key): item
        for item in data
        if item.get(key) is not None
    }.values())



In [25]:
data_path = 'data/full_anime_copy.json'
anime_dict = get_json_anime_data(filepath=data_path)
studios_data = get_json_studios_data(filepath=data_path)

anime_df = pd.DataFrame(anime_dict).drop(columns=["url", 'aired', 'explicit_genres'])

In [27]:
anime_df_nan = anime_df.dropna(axis=0)
anime_df.shape, anime_df_nan.shape

((5017, 18), (4215, 18))

In [29]:
anime_df.head()

,mal_id,title_english,type,source,episodes,duration,rating,score,scored_by,rank,popularity,members,favorites,year,demographics,studios,genres,themes
0,1,Cowboy Bebop,TV,Original,26.0,24 min per ep,R - 17+ (violence & profanity),8.75,1062212.0,49.0,41,2057323,89639,1998,[],[Sunrise],"[Action, Award Winning, Sci-Fi]","[Adult Cast, Space]"
1,5,Cowboy Bebop: The Movie,Movie,Original,1.0,1 hr 55 min,R - 17+ (violence & profanity),8.38,232423.0,240.0,659,412138,1785,2001,[],[Bones],"[Action, Sci-Fi]","[Adult Cast, Space]"
2,6,Trigun,TV,Manga,26.0,24 min per ep,PG-13 - Teens 13 or older,8.22,401751.0,424.0,266,834750,17659,1998,[Shounen],[Madhouse],"[Action, Adventure, Sci-Fi]",[Adult Cast]
3,7,Witch Hunter Robin,TV,Original,26.0,25 min per ep,PG-13 - Teens 13 or older,7.26,46713.0,3603.0,1993,129851,712,2002,[],[Sunrise],"[Action, Drama, Mystery, Supernatural]",[Detective]
4,8,Beet the Vandel Buster,TV,Manga,52.0,23 min per ep,PG - Children,7.05,7355.0,4849.0,5888,17035,18,2004,[Shounen],[Toei Animation],"[Action, Adventure, Fantasy]",[]


In [30]:
import re


def process_duration(duration_str):
    # 24 min per ep -> 24 min
    # 1 hr 55 min -> 115 min etc
    if pd.isna(duration_str):
        return None
    
    if not duration_str or duration_str.lower() == "unknown":
        return None

    hours = 0
    minutes = 0

    hr_match = re.search(r"(\d+)\s*hr", duration_str)
    min_match = re.search(r"(\d+)\s*min", duration_str)

    if hr_match:
        hours = int(hr_match.group(1))

    if min_match:
        minutes = int(min_match.group(1))

    return hours * 60 + minutes


def cols_to_int(df, cols):
    df_copy = df.copy()
    
    df_copy[cols] = (
        df_copy[cols]
        .apply(pd.to_numeric, errors="coerce")
        .astype("Int64")
    )
    return df_copy


def process_demographics(df):
    # nan -> []
    # [] -> []
    # not_list(x) -> list(x)
    df_copy = df.copy()
    
    def clean(x):
        if isinstance(x, (list, tuple, np.ndarray)):
            return list(x) if len(x) > 0 else []

        if isinstance(x, list):
            return x if len(x) > 0 else []

        return [x]
    
    df_copy["demographics"] = df_copy["demographics"].apply(clean)
    return df_copy


def process_anime_dataset(df):
    df_copy = cols_to_int(df, cols=[
        "episodes",
        "scored_by",
        "rank",
        "popularity",
        "members",
        "favorites",
        "year"
    ])
    
    df_copy["duration"] = (
        df_copy["duration"]
        .apply(process_duration)
        .astype("Int64")
    )
    
    df_copy = process_demographics(df_copy)
    
    return df_copy


In [31]:
new_anime_df = process_anime_dataset(anime_df)
new_anime_df.shape


(5017, 18)

In [34]:
new_anime_df.to_parquet("data/anime_processed.parquet", index=False)

new_anime_df.head(5)


,mal_id,title_english,type,source,episodes,duration,rating,score,scored_by,rank,popularity,members,favorites,year,demographics,studios,genres,themes
0,1,Cowboy Bebop,TV,Original,26,24,R - 17+ (violence & profanity),8.75,1062212,49,41,2057323,89639,1998,[],[Sunrise],"[Action, Award Winning, Sci-Fi]","[Adult Cast, Space]"
1,5,Cowboy Bebop: The Movie,Movie,Original,1,115,R - 17+ (violence & profanity),8.38,232423,240,659,412138,1785,2001,[],[Bones],"[Action, Sci-Fi]","[Adult Cast, Space]"
2,6,Trigun,TV,Manga,26,24,PG-13 - Teens 13 or older,8.22,401751,424,266,834750,17659,1998,[Shounen],[Madhouse],"[Action, Adventure, Sci-Fi]",[Adult Cast]
3,7,Witch Hunter Robin,TV,Original,26,25,PG-13 - Teens 13 or older,7.26,46713,3603,1993,129851,712,2002,[],[Sunrise],"[Action, Drama, Mystery, Supernatural]",[Detective]
4,8,Beet the Vandel Buster,TV,Manga,52,23,PG - Children,7.05,7355,4849,5888,17035,18,2004,[Shounen],[Toei Animation],"[Action, Adventure, Fantasy]",[]


In [35]:
parquet_df = pd.read_parquet("data/anime_processed.parquet")
parquet_df.head(5)


,mal_id,title_english,type,source,episodes,duration,rating,score,scored_by,rank,popularity,members,favorites,year,demographics,studios,genres,themes
0,1,Cowboy Bebop,TV,Original,26,24,R - 17+ (violence & profanity),8.75,1062212,49,41,2057323,89639,1998,[],[Sunrise],"[Action, Award Winning, Sci-Fi]","[Adult Cast, Space]"
1,5,Cowboy Bebop: The Movie,Movie,Original,1,115,R - 17+ (violence & profanity),8.38,232423,240,659,412138,1785,2001,[],[Bones],"[Action, Sci-Fi]","[Adult Cast, Space]"
2,6,Trigun,TV,Manga,26,24,PG-13 - Teens 13 or older,8.22,401751,424,266,834750,17659,1998,[Shounen],[Madhouse],"[Action, Adventure, Sci-Fi]",[Adult Cast]
3,7,Witch Hunter Robin,TV,Original,26,25,PG-13 - Teens 13 or older,7.26,46713,3603,1993,129851,712,2002,[],[Sunrise],"[Action, Drama, Mystery, Supernatural]",[Detective]
4,8,Beet the Vandel Buster,TV,Manga,52,23,PG - Children,7.05,7355,4849,5888,17035,18,2004,[Shounen],[Toei Animation],"[Action, Adventure, Fantasy]",[]
